# **Import**


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from data_loader import Dataset, standardize
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn

# **Build From Scratch**


In [2]:
class Linear:
    def __init__(self, input_size, num_neurons):

        self.W = np.random.randn(input_size, num_neurons)
        self.b = np.random.randn(1, num_neurons)

    def forward(self, X):

        self.X = X

        return self.X @ self.W + self.b

    def backward(self, glued_domino, learning_rate):

        dW = self.X.T @ glued_domino
        db = np.sum(glued_domino, axis=0, keepdims=True)

        glued_domino_new = glued_domino @ self.W.T

        self.W -= learning_rate * dW
        self.b -= learning_rate * db

        return glued_domino_new

class ReLU:

    def forward(self, X):

        self.mask = (X > 0).astype(float)

        return X * self.mask

    def backward(self, glued_domino, learning_rate):
        
        return glued_domino * self.mask

In [3]:
class Sequential:
    def __init__(self, *building_blocks):
        
        self.building_blocks = building_blocks

    def forward(self, X):

        for block in self.building_blocks:
            X = block.forward(X)

        return X

    def backward(self, glued_domino, learning_rate):

        for block in reversed(self.building_blocks):
            glued_domino = block.backward(glued_domino, learning_rate)

In [4]:
class MSELoss:
    def forward(self, y_pred, y):
        y = y.reshape(-1, 1)

        return np.mean((y_pred - y) ** 2)

    def backward(self, y_pred, y):
        y = y.reshape(-1, 1)
        
        return 2 * (y_pred - y) / y.shape[0]

In [5]:
class Sigmoid:
    def forward(self, X):
        self.out = 1 / (1 + np.exp(-X))
        return self.out

    def backward(self, glued_domino, learning_rate):
        return glued_domino * (self.out * (1 - self.out))

In [6]:
class BCELoss:
    def forward(self, y_pred, y):
        y = y.reshape(-1, 1)
        return -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))

    def backward(self,  y_pred, y):
        y = y.reshape(-1, 1)
        return (y_pred - y) / ( y_pred * (1 - y_pred) * y.shape[0])

In [7]:
class Softmax:
    def forward(self, X):
        exps = np.exp(X)
        self.out = exps / np.sum(exps, axis=1, keepdims=True)
        return self.out

    def backward(self, glued_domino, learning_rate):
        dot = np.sum(glued_domino * self.out, axis=1, keepdims=True)
        return self.out * (glued_domino - dot)

class CrossEntropyLoss:
    def forward(self, y_pred, y):
        y = y.reshape(-1, 1)
        losses = []
        for i in range(len(y)):
            losses.append(-np.log(y_pred[i, y[i]]))
        return np.mean(losses)

    def backward(self, y_pred, y):
        y = y.reshape(-1, 1)
        gradient = y_pred.copy()
        for i in range(len(y)):
            gradient[i, y[i]] -= 1
        return gradient / len(y)

In [8]:
def train(model, objective, X_train, y_train, num_iterations=1000, learning_rate=0.001):

    for i in range(num_iterations):

        y_pred = model.forward(X_train)
        cost = objective.forward(y_pred, y_train)

        gradient = objective.backward(y_pred, y_train)
        model.backward(gradient, learning_rate)

        if (i + 1) % 100 == 0:
            print(f"{cost:.3f}")

# **Load, Split and Standardize**

In [9]:
dataset_r = Dataset("regression")
X_train_r, X_test_r, y_train_r, y_test_r = dataset_r.load_split_data()
X_train_r, X_test_r = standardize(X_train_r, X_test_r)

dataset_b = Dataset("binary classification")
X_train_b, X_test_b, y_train_b, y_test_b = dataset_b.load_split_data()
X_train_b, X_test_b = standardize(X_train_b, X_test_b)

dataset_m = Dataset("multiclass classification")
X_train_m, X_test_m, y_train_m, y_test_m = dataset_m.load_split_data()
X_train_m, X_test_m = standardize(X_train_m, X_test_m)

# **Train, Test and Compare**

In [10]:
num_features = X_train_r.shape[1]

custom_model_r = Sequential(
    Linear(num_features, 3),
    ReLU(),
    Linear(3, 1)
)

custom_objective_r = MSELoss()

train(custom_model_r, custom_objective_r, X_train_r, y_train_r, num_iterations=1500, learning_rate=0.01)

1.094
0.779
0.656
0.594
0.560
0.538
0.523
0.512
0.504
0.499
0.494
0.491
0.488
0.486
0.485


In [11]:
num_features = X_train_b.shape[1]

custom_model_b = Sequential(
    Linear(num_features, 2),
    ReLU(),
    Linear(2, 3),
    Sigmoid(),
    Linear(3, 1),
    Sigmoid()
)

custom_objective_b = BCELoss()

train(custom_model_b, custom_objective_b, X_train_b, y_train_b, num_iterations=10000, learning_rate=0.01)

0.628
0.617
0.608
0.598
0.589


0.579
0.569
0.559
0.548
0.537
0.525
0.513
0.501
0.489
0.476
0.463
0.450
0.437
0.424
0.411
0.399
0.388
0.378
0.368
0.359
0.350
0.341
0.333
0.325
0.317
0.310
0.303
0.296
0.290
0.284
0.279
0.273
0.268
0.262
0.257
0.252
0.247
0.242
0.236
0.231
0.227
0.222
0.217
0.213
0.209
0.205
0.201
0.198
0.194
0.191
0.188
0.186
0.183
0.181
0.178
0.176
0.174
0.172
0.170
0.168
0.166
0.165
0.163
0.162
0.160
0.159
0.157
0.156
0.155
0.154
0.152
0.151
0.150
0.149
0.148
0.147
0.146
0.146
0.145
0.144
0.143
0.142
0.142
0.141
0.140
0.140
0.139
0.138
0.138
0.137
0.137
0.136
0.135
0.135
0.134


In [12]:
num_features = X_train_m.shape[1]
num_classes = len(set(y_train_m))

custom_model_m = Sequential(
    Linear(num_features, 2),
    ReLU(),
    Linear(2, 3),
    Sigmoid(),
    Linear(3, num_classes),
    Softmax()
)

custom_objective_m = CrossEntropyLoss()

train(custom_model_m, custom_objective_m, X_train_m, y_train_m, num_iterations=20000, learning_rate=0.05)

1.105
1.003
0.896
0.809
0.746
0.699
0.658
0.622
0.587
0.555
0.524
0.496
0.468
0.442
0.416
0.392
0.369
0.348
0.328
0.310
0.294
0.279
0.266
0.254
0.243
0.233
0.224
0.216
0.209
0.203
0.197
0.191
0.186
0.182
0.177
0.173
0.170
0.166
0.163
0.160
0.157
0.154
0.152
0.150
0.147
0.145
0.143
0.141
0.139
0.138
0.136
0.134
0.133
0.131
0.130
0.129
0.127
0.126
0.125
0.124
0.123
0.122
0.120
0.119
0.118
0.118
0.117
0.116
0.115
0.114
0.113
0.112
0.112
0.111
0.110
0.110
0.109
0.108
0.108
0.107
0.106
0.106
0.105
0.105
0.104
0.103
0.103
0.102
0.102
0.101
0.101
0.100
0.100
0.100
0.099
0.099
0.098
0.098
0.097
0.097
0.097
0.096
0.096
0.095
0.095
0.095
0.094
0.094
0.094
0.093
0.093
0.093
0.092
0.092
0.092
0.092
0.091
0.091
0.091
0.090
0.090
0.090
0.090
0.089
0.089
0.089
0.089
0.088
0.088
0.088
0.088
0.087
0.087
0.087
0.087
0.087
0.086
0.086
0.086
0.086
0.085
0.085
0.085
0.085
0.085
0.084
0.084
0.084
0.084
0.084
0.084
0.083
0.083
0.083
0.083
0.083
0.083
0.082
0.082
0.082
0.082
0.082
0.082
0.081
0.081
0.081
0.08

In [13]:
y_pred_custom_r = custom_model_r.forward(X_test_r)

print(f"Custom Regression MSE: {mean_squared_error(y_test_r, y_pred_custom_r):.3f}")

y_pred_custom_prob = custom_model_b.forward(X_test_b)
y_pred_custom_b = (y_pred_custom_prob.flatten() > 0.5).astype(int)

print(f"Custom Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_custom_b):.3f}")

y_pred_custom_prob = custom_model_m.forward(X_test_m)
y_pred_custom_m = np.argmax(y_pred_custom_prob, axis=1)

print(f"Custom Multiclass Classification Accuracy: {accuracy_score(y_test_m, y_pred_custom_m):.3f}")


Custom Regression MSE: 0.494
Custom Binary Classification Accuracy: 0.985
Custom Multiclass Classification Accuracy: 1.000


In [14]:
X_train_r_tensor = torch.tensor(X_train_r, dtype=torch.float32)
y_train_r_tensor = torch.tensor(y_train_r, dtype=torch.float32).view(-1, 1)
X_test_r_tensor = torch.tensor(X_test_r, dtype=torch.float32)
y_test_r_tensor = torch.tensor(y_test_r, dtype=torch.float32).view(-1, 1)

X_train_b_tensor = torch.tensor(X_train_b, dtype=torch.float32)
y_train_b_tensor = torch.tensor(y_train_b, dtype=torch.float32).view(-1, 1)
X_test_b_tensor = torch.tensor(X_test_b, dtype=torch.float32)
y_test_b_tensor = torch.tensor(y_test_b, dtype=torch.float32).view(-1, 1)

X_train_m_tensor = torch.tensor(X_train_m, dtype=torch.float32)
y_train_m_tensor = torch.tensor(y_train_m, dtype=torch.long)
X_test_m_tensor = torch.tensor(X_test_m, dtype=torch.float32)
y_test_m_tensor = torch.tensor(y_test_m, dtype=torch.long)

In [15]:
num_features = X_train_r.shape[1]

pytorch_model_r = nn.Sequential(
    nn.Linear(num_features, 3),
    nn.ReLU(),
    nn.Linear(3, 1)
)

pytorch_objective_r = nn.MSELoss()

def train_pytorch(model, objective, X_train, y_train, num_iterations=1000, learning_rate=0.001):

    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    for i in range(num_iterations):

        y_pred = model(X_train)

        cost = objective(y_pred, y_train)

        optimizer.zero_grad()

        cost.backward()
        optimizer.step()

        if (i + 1) % 100 == 0:
            print(f"{cost.item():.3f}")

train_pytorch(pytorch_model_r, pytorch_objective_r, X_train_r_tensor, y_train_r_tensor, num_iterations=1500, learning_rate=0.01)

0.819
0.603
0.540
0.506
0.485
0.471
0.462
0.456
0.453
0.450
0.449
0.447
0.446
0.445
0.444


In [16]:
num_features = X_train_b.shape[1]

pytorch_model_b = nn.Sequential(
    nn.Linear(num_features, 2),
    nn.ReLU(),
    nn.Linear(2, 3),
    nn.Sigmoid(),
    nn.Linear(3, 1),
    nn.Sigmoid()
)

pytorch_objective_b = nn.BCELoss()

train_pytorch(pytorch_model_b, pytorch_objective_b, X_train_b_tensor, y_train_b_tensor, num_iterations=10000, learning_rate=0.01)

0.663
0.656
0.652
0.650
0.649
0.648
0.647
0.646
0.645
0.644
0.643
0.642
0.640
0.638
0.636
0.633
0.629
0.624
0.618
0.610
0.600
0.586
0.569
0.548
0.523
0.494
0.463
0.430
0.398
0.367
0.338
0.312
0.289
0.269
0.252
0.237
0.224
0.212
0.202
0.193
0.185
0.179
0.173
0.167
0.162
0.158
0.154
0.150
0.147
0.144
0.141
0.139
0.137
0.134
0.133
0.131
0.129
0.127
0.126
0.125
0.123
0.122
0.121
0.119
0.118
0.117
0.116
0.115
0.114
0.113
0.113
0.112
0.111
0.110
0.110
0.109
0.108
0.108
0.107
0.106
0.106
0.105
0.105
0.104
0.104
0.103
0.103
0.102
0.102
0.102
0.101
0.101
0.100
0.100
0.100
0.099
0.099
0.099
0.098
0.098


In [17]:
num_features = X_train_m.shape[1]
num_classes = len(set(y_train_m))

pytorch_model_m = nn.Sequential(
    nn.Linear(num_features, 2),
    nn.ReLU(),
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Linear(3, num_classes)
)

pytorch_objective_m = nn.CrossEntropyLoss()

train_pytorch(pytorch_model_m, pytorch_objective_m, X_train_m_tensor, y_train_m_tensor, num_iterations=20000, learning_rate=0.05)

1.097
1.096
1.096
1.096
1.096
1.096
1.096
1.096
1.096
1.096
1.096
1.096
1.096
1.096
1.096
1.095
1.034
0.853
0.659
0.567
0.507
0.447
0.389
0.339
0.298
0.264
0.236
0.213
0.193
0.177
0.163
0.151
0.141
0.132
0.124
0.117
0.111
0.106
0.101
0.096
0.093
0.089
0.086
0.083
0.081
0.078
0.076
0.074
0.073
0.071
0.070
0.068
0.067
0.066
0.065
0.064
0.063
0.062
0.062
0.061
0.060
0.060
0.059
0.059
0.058
0.058
0.057
0.057
0.056
0.056
0.056
0.055
0.055
0.055
0.054
0.054
0.054
0.054
0.053
0.053
0.053
0.053
0.053
0.052
0.052
0.052
0.052
0.052
0.051
0.051
0.051
0.051
0.051
0.051
0.051
0.050
0.050
0.050
0.050
0.050
0.050
0.050
0.050
0.050
0.050
0.050
0.050
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.049
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.048
0.04

In [18]:
y_pred_tensor = pytorch_model_r(X_test_r_tensor)

y_pred_pytorch = y_pred_tensor.detach().numpy()

print(f"PyTorch Regression MSE: {mean_squared_error(y_pred_pytorch, y_test_r):.3f}")

y_pred_pytorch = pytorch_model_b.forward(X_test_b_tensor).detach().numpy()
y_pred_pytorch = (y_pred_pytorch.flatten() > 0.5).astype(int)

print(f"PyTorch Binary Classification Accuracy: {accuracy_score(y_pred_pytorch, y_test_b):.3f}")

y_pred_pytorch_prob = pytorch_model_m.forward(X_test_m_tensor).detach().numpy()

y_pred_pytorch = np.argmax(y_pred_pytorch_prob, axis=1)

print(f"PyTorch Multiclass Classification Accuracy: {accuracy_score(y_pred_pytorch, y_test_m):.3f}")

PyTorch Regression MSE: 0.461
PyTorch Binary Classification Accuracy: 1.000
PyTorch Multiclass Classification Accuracy: 1.000
